In [1]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
from nn_utils import split_and_shuffle, combine, train_pca_pipeline, predict_pca_pipeline, build_pca_features
import os
from global_vars import *


train_folder = os.path.join(data_folder, 'nodeep_train_data_ver1/')
test_folder = os.path.join(data_folder, 'nodeep_test_data_ver1/')


X_train_val = pd.read_parquet(os.path.join(train_folder, "X_train.parquet"),engine="fastparquet")
y_t_train_val = pd.read_parquet(os.path.join(train_folder, "y_t_train.parquet"),engine="fastparquet")
y_q_train_val = pd.read_parquet(os.path.join(train_folder, "y_q_train.parquet"),engine="fastparquet")

X_test = pd.read_parquet(os.path.join(test_folder, "X_test.parquet"),engine="fastparquet")
y_t_test = pd.read_parquet(os.path.join(test_folder, "y_t_test.parquet"),engine="fastparquet")
y_q_test = pd.read_parquet(os.path.join(test_folder, "y_q_test.parquet"),engine="fastparquet")

In [2]:
X_train, y_train, X_val, y_val = split_and_shuffle(
    X_train_val.values, y_t_train_val.values, val_frac=0.2, seed=0
)
n_surf_features = len(surf_input_var_names)
n_total_features = X_train.shape[1]
branch2_start = (n_total_features-n_surf_features)
Xa_train = X_train[:, range(branch2_start)]
Xb_train = X_train[:, range(branch2_start, n_total_features)]
Xa_val = X_val[:, range(branch2_start)]
Xb_val = X_val[:, range(branch2_start, n_total_features)]

In [6]:
lsts = [
    (256, 128, 64 ),
    (512, 256, 128 ),
    (128, 64, 32 ),
    (64, 32, 16 ),
]


for lst in lsts:
    print(lst)


    model, scaler, history, pca_objs = train_pca_pipeline(
        Xa_train, Xb_train, y_train,
        Xa_val=Xa_val, Xb_val=Xb_val, y_val=y_val,
        n_components=12, #50, #24
        epochs=200,
        mlp_args={"hidden_sizes": lst, "dropout": [0.2, 0.1, 0.1]},
        weight_decay=1e-4,
        patience=20,
    )

(256, 128, 64)
PCA: 232 -> 12 components, retaining 89.4% of branch-1 variance
combined feature count: 17 (12 PCA + 5 raw)
epoch   0 | train 0.7211 | val 0.5845
epoch   1 | train 0.6918 | val 0.5766
epoch   2 | train 0.6815 | val 0.5801
epoch   3 | train 0.6785 | val 0.5642
epoch   4 | train 0.6713 | val 0.5649
epoch   5 | train 0.6703 | val 0.5582
epoch   6 | train 0.6665 | val 0.5674
epoch   7 | train 0.6618 | val 0.5577
epoch   8 | train 0.6632 | val 0.5559
epoch   9 | train 0.6635 | val 0.5507
epoch  10 | train 0.6574 | val 0.5556
epoch  11 | train 0.6581 | val 0.5543
epoch  12 | train 0.6560 | val 0.5516
epoch  13 | train 0.6534 | val 0.5508
epoch  14 | train 0.6519 | val 0.5458
epoch  15 | train 0.6511 | val 0.5497
epoch  16 | train 0.6509 | val 0.5426
epoch  17 | train 0.6511 | val 0.5461
epoch  18 | train 0.6505 | val 0.5589
epoch  19 | train 0.6480 | val 0.5452
epoch  20 | train 0.6488 | val 0.5470
epoch  21 | train 0.6466 | val 0.5500
epoch  22 | train 0.6439 | val 0.5560
epo